# The ablation, under the leakage-free protocol

Every published ablation number came from `08_train_colab.ipynb`, which runs the
**older protocol** — features computed over the whole graph, so a training-window
feature could contain information from the future. Those numbers are inflated and
cannot be compared with the served model, which was trained temporally.

This notebook retrains every stage the same way the served model was trained, so the
ablation and the deployed system are finally measured on the same footing.

| stage | what it adds |
|---|---|
| 1  | BaselineGraphSAGE, BCE + pos_weight |
| 2  | + Edge-MLP attention (Novelty 1) |
| 3a | + Focal loss (Novelty 2, loss half) |
| 3b | + Graph-aware sampler (Novelty 2, sampling half) — **this is what ships** |
| 3c | 3b **minus** the Edge-MLP — the leave-one-out arm for Novelty 1 |

**3c is the one that matters most.** Stage 2 vs 1 measures edge attention in
isolation, where it shows almost nothing. 3b vs 3c measures it inside the full
system, which is the only comparison that supports the claim.

Runs are **resumable**: a stage whose report is already on Drive is skipped, so a
Colab disconnect costs you one stage, not the whole run.


## 1 · Environment


In [ ]:
import os, pathlib, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

DRIVE  = pathlib.Path('/content/drive/MyDrive/deepsentinel')
RUNS   = DRIVE / 'temporal_ablation'      # results land here, survive a disconnect
RUNS.mkdir(parents=True, exist_ok=True)

REPO = pathlib.Path('/content/Graphsage')
if not REPO.exists():
    !git clone -q https://github.com/R26-IT-121/Graphsage.git {REPO}
%cd {REPO}
!git pull -q
!pip -q install -e . 2>&1 | tail -2

import torch
print('device:', 'cuda' if torch.cuda.is_available() else 'CPU  <-- check Runtime > Change runtime type')


## 2 · The feature table

`features.parquet` (65 MB) is the only input. Kept on Drive so it is built once.


In [ ]:
proc = REPO / 'data' / 'processed'; proc.mkdir(parents=True, exist_ok=True)
src = DRIVE / 'features.parquet'
if not (proc / 'features.parquet').exists():
    if src.exists():
        !cp {src} {proc}/features.parquet
    else:
        !python scripts/download_paysim.py && python scripts/prepare_features.py
        !cp {proc}/features.parquet {src}
print('features.parquet:', (proc / 'features.parquet').stat().st_size / 1e6, 'MB')


## 3 · The temporal graph

`--features v2` gives the 12 behavioural node features the served model uses.
Cached to Drive — building it takes a while and never changes.


In [ ]:
graph = REPO / 'data' / 'graph' / 'paysim_temporal_v2.pt'
graph.parent.mkdir(parents=True, exist_ok=True)
cached = DRIVE / 'paysim_temporal_v2.pt'

if cached.exists():
    !cp {cached} {graph}
else:
    !python scripts/build_temporal_graph.py --features v2
    !cp {graph} {cached}
print('graph:', graph.stat().st_size / 1e6, 'MB')


## 4 · Train every stage

Seed 0 first — that alone gives you the ablation table. Extra seeds come next,
and are what turn 'Novelty 1 helps' into a claim with a confidence interval.

Each stage copies its report, scores and checkpoint to Drive the moment it
finishes, so nothing is lost if the session drops.


In [ ]:
STAGES = ['1', '2', '3a', '3b', '3c']
SEEDS  = [0]          # add 1, 2 after every stage has a seed-0 result
EPOCHS = 50

def done(stage, seed):
    return (RUNS / f'stage{stage}_v2_seed{seed}.json').exists()

for seed in SEEDS:
    for stage in STAGES:
        if done(stage, seed):
            print(f'skip  stage {stage} seed {seed} — already on Drive'); continue
        print(f'\n=== stage {stage}, seed {seed} ' + '='*40)
        rc = subprocess.call([sys.executable, 'scripts/train_temporal.py',
                              '--stage', stage, '--seed', str(seed),
                              '--features', 'v2', '--epochs', str(EPOCHS)])
        if rc != 0:
            print(f'stage {stage} seed {seed} FAILED (exit {rc}) — continuing'); continue
        tag = f'stage{stage}_v2_seed{seed}'
        for pat in (f'reports/temporal/{tag}.json',
                    f'reports/temporal/{tag}_scores.pt',
                    f'checkpoints/temporal_{tag}.pt'):
            p = REPO / pat
            if p.exists():
                !cp {p} {RUNS}/
        print('saved to Drive:', tag)


## 5 · The table

This is the figure for the final presentation. Read 3b vs 3c for Novelty 1, and
3b vs 3a for the sampler half of Novelty 2.


In [ ]:
import json, pandas as pd

rows = []
for f in sorted(RUNS.glob('stage*_v2_seed*.json')):
    d = json.load(open(f))
    t = d.get('test') or d.get('tuned_threshold_metrics', {}).get('test', {})
    v = d.get('val')  or d.get('tuned_threshold_metrics', {}).get('val', {})
    rows.append({
        'stage': d.get('stage'), 'seed': d.get('seed'),
        'threshold': round(d.get('best_threshold', float('nan')), 4),
        'val F1':  round(v.get('f1', float('nan')), 4),
        'test P':  round(t.get('precision', float('nan')), 4),
        'test R':  round(t.get('recall', float('nan')), 4),
        'test F1': round(t.get('f1', float('nan')), 4),
        'test AUC':round(t.get('auroc', float('nan')), 4),
    })

df = pd.DataFrame(rows).sort_values(['stage', 'seed'])
print(df.to_string(index=False))
df.to_csv(RUNS / 'ablation_temporal.csv', index=False)
print('\nwrote', RUNS / 'ablation_temporal.csv')


## 6 · What to check before you trust it

- Every stage reports `protocol: temporal_snapshots_leakage_free`. If one does not,
  it did not run the pipeline you think it did.
- Test AUC should land near **0.70**, not 0.94. The old numbers were ~0.94 *because*
  of the leak — a high number here means the graph was built the old way.
- 3b's numbers should be close to the served model (precision 0.668, recall 0.271,
  F1 0.386). If they are far off, the served bundle and this run disagree and the
  bundle should be re-exported from this checkpoint.

Then download `ablation_temporal.csv` and the reports, and commit the reports to
`reports/temporal/` in the repo.
